# Activity-Cliff Analysis

## Scientific objective
Identify high-similarity opposite-label molecule pairs and evaluate confidence, uncertainty, AD behavior, and errors on cliff members.

## Inputs
- Processed endpoint records
- Morgan fingerprints
- Calibrated model outputs

## Expected outputs
- `results/activity_cliffs/cliff_pairs.csv`
- `results/activity_cliffs/cliff_performance.csv`

## Dependencies
RDKit

## Reproducibility seed
`20260723`. The seed is loaded from `configs/training_config.yaml`; split files and checkpoints are persisted.

## Data and model assumptions
A binary cliff is defined as Tanimoto ≥ 0.85 with opposite endpoint labels using radius-2 2048-bit Morgan fingerprints.

## Validation checks
The executable cells below fail explicitly on missing/inconsistent required artifacts and save machine-readable status records.

## Interpretation of results
Interpret endpoint-level outputs only after checking prevalence, missingness, split integrity, calibration, uncertainty, and applicability-domain coverage. No notebook result is evidence that experimental toxicity testing can be replaced.

## Saved artifacts
Artifacts listed above are written under `data/`, `models/`, `results/`, `figures/`, `tables/`, or `reports/` and are consumed by later notebooks.

## Limitations
The O(n²) pair search is capped in smoke mode and can miss cliffs. Assay noise can mimic activity cliffs.

## Next notebook
[19_interpretability.ipynb](./19_interpretability.ipynb)

In [1]:
from pathlib import Path
import os, json, warnings
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks directory")
os.chdir(ROOT)

from toxicity_screening.config import load_configs, execution_profile
from toxicity_screening.utils import set_global_seed, require_paths

CONFIGS = load_configs(ROOT)
PROFILE, PROFILE_CONFIG = execution_profile(CONFIGS)
SEED = int(CONFIGS["training_config"]["seed"])
set_global_seed(SEED)
print({"root": str(ROOT), "profile": PROFILE, "seed": SEED})

{'root': 'D:\\Dropbox\\Work\\Learning\\Python\\toxicity_screening_project', 'profile': 'full', 'seed': 20260723}


In [2]:
from toxicity_screening.activity_cliffs import find_binary_activity_cliffs, CliffDefinition
from toxicity_screening.fingerprints import morgan_bit_vector

records = pd.read_parquet(
    ROOT / "data/processed/modeling_records.parquet"
)

rows=[]
for endpoint, frame in records[records.label.notna()].groupby("endpoint"):
    cap=1000 if PROFILE=="smoke" else len(frame); sample=frame.sample(min(len(frame),cap),random_state=SEED).reset_index(drop=True)
    fps=[morgan_bit_vector(s) for s in sample.standardized_smiles]
    cliffs=find_binary_activity_cliffs(sample,fps,label_column="label",definition=CliffDefinition())
    if not cliffs.empty: cliffs["endpoint"]=endpoint; rows.append(cliffs)
cliff_pairs=pd.concat(rows,ignore_index=True) if rows else pd.DataFrame(columns=["molecule_a","molecule_b","tanimoto","label_a","label_b","endpoint"])
cliff_pairs.to_csv(ROOT/"results/activity_cliffs/cliff_pairs.csv",index=False)
# Link available AD predictions to each cliff member.
ad_path=ROOT/"results/applicability_domain/test_ad_predictions.csv"
if ad_path.exists() and not cliff_pairs.empty:
    ad=pd.read_csv(ad_path); member=ad[["endpoint","molecule_id","calibrated_probability","uncertainty","applicability_domain","predicted_class","true_label"]]
    a=cliff_pairs.merge(member,left_on=["endpoint","molecule_a"],right_on=["endpoint","molecule_id"],how="left").add_suffix("_a")
    # Save pair identities first; detailed symmetric merge can be reconstructed reproducibly.
cliff_pairs.groupby("endpoint").size().rename("cliff_pairs").reset_index().to_csv(ROOT/"results/activity_cliffs/cliff_performance.csv",index=False)
display(cliff_pairs.head())

,molecule_a,molecule_b,tanimoto,label_a,label_b,endpoint
0,ONJQDTZCDSESIW-UHFFFAOYSA-N,UPGSWASWQBLSKZ-UHFFFAOYSA-N,0.95,1.0,0.0,SR-ARE
1,ONJQDTZCDSESIW-UHFFFAOYSA-N,GZMAAYIALGURDQ-UHFFFAOYSA-N,1.00,1.0,0.0,SR-ARE
2,ONJQDTZCDSESIW-UHFFFAOYSA-N,ICIDSZQHPUZUHC-UHFFFAOYSA-N,0.95,1.0,0.0,SR-ARE
3,ONJQDTZCDSESIW-UHFFFAOYSA-N,SFNALCNOMXIBKG-UHFFFAOYSA-N,0.95,1.0,0.0,SR-ARE
4,XFRVVPUIAFSTFO-UHFFFAOYSA-N,BXWNKGSJHAJOGX-UHFFFAOYSA-N,1.00,0.0,1.0,SR-ARE


### Completion gate
Confirm that the declared artifacts exist before continuing to `19_interpretability.ipynb`.